# RacketVision -> 2D racket keypoints (Stage 1)  [v4]

**Goal:** run the pretrained **RacketVision** racket-pose model (RTMDet detector ->
RTMPose) on a badminton clip and emit **per-frame 2D keypoints** + an overlay video.

Keypoint order (confirmed from the repo's dataset config):
`0=top, 1=bottom, 2=handle, 3=left, 4=right`. The racket long axis = `handle`(grip) ->
`top`(head tip); `left`/`right` = head width.

**RUNTIME: set GPU first** - Runtime -> Change runtime type -> **T4 GPU**. (CPU works but is slow.)

**Why the pinned versions:** Colab ships Python 3.12 + torch 2.11, and OpenMMLab (mmcv)
has no wheels for that. Cell 2 pins **torch 2.3.1+cu121** and installs mmcv/mmdet/mmpose
with `--no-deps` so they don't fight over versions. Repo: github.com/OrcustD/RacketVision (MIT).

**Run order:** Cell 1 -> **Cell 2** (installs, then it *restarts the kernel on purpose* --
that's expected, not a crash) -> after the restart, **Cell 2c** (verifies + auto-repairs
numpy, probes every import) -> **Cells 3-8**. Do NOT re-run Cell 2 after the restart.

> Cell 2 is the fragile part. **Cell 2c is the debug cell**: it repairs numpy in-kernel and
> imports each heavy module one at a time, printing PASS/FAIL. If anything FAILs, paste me
> its whole output -- that names the culprit without a full Cell-3->5 re-run.


In [ ]:
# --- Cell 1: runtime check (do NOT import torch yet) ---------------------
import sys
print("Python:", sys.version)
!nvidia-smi -L || echo ">>> NO GPU. Runtime > Change runtime type > T4 GPU, then rerun. (CPU works but is slow.)"


In [ ]:
# --- Cell 2: ONE-TIME SETUP (GPU runtime) -------------------------------
# This cell RESTARTS the kernel at the end -- ON PURPOSE. The Colab kernel imports
# numpy 2.x at startup, so the clean numpy 1.26.4 installed below can only take effect
# in a fresh kernel. That is expected, not a crash. AFTER it restarts: run Cell 2c
# (verify + auto-repair), then Cells 3-8. Do NOT re-run Cell 2.
#
# Why the numpy dance: mmcv 2.2.0 is the ONLY py3.12/torch2.3 prebuilt wheel, and it's
# compiled against numpy 1.x -- so the WHOLE stack must stay on numpy 1.26.4. Colab ships
# numpy 2.x and several installs try to pull it back to 2.x -> ABI crash at import
# ("numpy.dtype size changed, Expected 96 ... got 88"). FIX: a pip *constraints file*
# pins numpy==1.26.4 for EVERY install below, so nothing can ever bounce it to 2.x.

# constraints file -> passed to every pip install via {C}
with open("/content/constraints.txt", "w") as _f:
    _f.write("numpy==1.26.4\n")
C = "-c /content/constraints.txt"

# 0) install a CLEAN numpy 1.26.4 FIRST so every compiled pkg below agrees on it
!pip uninstall -y numpy
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy /usr/local/lib/python3.12/dist-packages/numpy-*.dist-info /usr/local/lib/python3.12/dist-packages/numpy.libs
!pip -q install --no-cache-dir --force-reinstall --no-deps "numpy==1.26.4"
# 1) modern setuptools (py3.12: fixes pkgutil.ImpImporter + pkg_resources find_module)
!pip -q install {C} -U pip "setuptools>=70" wheel
# 2) a torch OpenMMLab has wheels for (Colab's torch 2.11 has none). cp312 + cu121.
!pip -q install {C} torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
# 3) mmcv PREBUILT wheel for torch2.3/cu121/cp312 (avoids a 20-min source build)
!pip -q install {C} -U openmim
!pip -q install {C} "mmengine>=0.10.7"
!pip -q install {C} mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3.0/index.html
# 4) mmdet + mmpose with --no-deps so they don't downgrade mmcv; + the few top-down deps
!pip -q install {C} "mmdet==3.3.0" --no-deps
!pip -q install {C} "mmpose==1.3.2" --no-deps
!pip -q install {C} pycocotools shapely terminaltables json_tricks munkres
# 4b) REMOVE transformers. Building ANY mmdet detector imports mmdet.models, which pulls in
#     its GLIP/Grounding-DINO language models -> `import transformers`. Colab's transformers
#     needs torch>=2.4; on torch 2.3.1 it disables its torch integration and then dies with
#     `NameError: name 'nn' is not defined` -- a NameError, which mmdet's `except ImportError`
#     does NOT catch, so init_detector() blows up. RTMDet is pure CNN and needs none of it;
#     with transformers absent mmdet just warns and moves on.
!pip uninstall -y transformers
# 5) scipy/opencv that match numpy 1.26 (the constraint keeps numpy pinned at 1.26.4)
!pip -q install {C} --no-cache-dir "scipy==1.12.0" "opencv-python==4.10.0.84"
# NOTE: xtcocotools won't build on py3.12 -> Cell 5 registers a stub (only the NAME is needed).

# 6) relax the mmcv version ceiling (disk edit, survives the restart below).
#    mmdet 3.3.0 / mmpose 1.3.2 hard-assert mmcv < 2.2.0, but py3.12 has NO prebuilt mmcv < 2.2.0.
#    They run fine with mmcv 2.2.0 for inference -- bump the ceiling so Cell 2c/Cell 5 imports pass.
import importlib.util, re
def _bump(name):
    ip = importlib.util.find_spec(name).origin          # locate __init__.py WITHOUT importing it
    s = open(ip).read(); orig = s
    s = re.sub(r"(mmcv_maximum_version\s*=\s*['\"])2\.2\.0(['\"])",  r"\g<1>2.3.0\g<2>", s)
    s = re.sub(r"(mmdet_maximum_version\s*=\s*['\"])3\.3\.0(['\"])", r"\g<1>3.4.0\g<2>", s)
    if s != orig: open(ip, "w").write(s)
    print(" ceiling:", name, "->", "patched" if s != orig else "no change")
for _n in ("mmdet", "mmpose"):
    _bump(_n)

# report the on-disk numpy WITHOUT importing it into this kernel, then restart clean
import subprocess, sys
_v = subprocess.run([sys.executable, "-c",
        "import numpy,os;print(numpy.__version__,'@',os.path.dirname(numpy.__file__))"],
        capture_output=True, text=True).stdout.strip()
print("\non-disk numpy ->", _v, "(want 1.26.4)")
print("*** Restarting the kernel now to load a clean numpy (EXPECTED, not a crash). ***")
print("*** After restart: run Cell 2c to verify, then Cells 3-8. Do NOT re-run Cell 2. ***")
import IPython; IPython.get_ipython().kernel.do_shutdown(True)   # if it doesn't auto-restart: Runtime > Restart session


In [ ]:
# --- Cell 2c: VERIFY + AUTO-REPAIR + DEBUG PROBE (run FIRST after the restart) ---
# Does three things, in order:
#   1. reports what numpy this kernel sees vs what's on disk (a mismatch = a stale kernel);
#   2. REPAIRS numpy to 1.26.4 in place and reloads it -- no second restart needed;
#   3. applies the same py3.12 shims as Cell 5, then imports each heavy module ONE AT A
#      TIME so a failure names the exact culprit instead of a 30-line chained traceback.
# Every line PASS -> Cells 3-8 will work. Any FAIL -> paste this whole output.
import importlib, importlib.machinery as _im, glob, os, pkgutil, subprocess, sys, types, traceback

WANT = "1.26.4"
def _sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout

print("numpy in THIS kernel :", sys.modules["numpy"].__version__ if "numpy" in sys.modules else "(not imported yet)")
print("numpy ON DISK        :", subprocess.run([sys.executable, "-c",
      "import numpy,os;print(numpy.__version__, os.path.dirname(numpy.__file__))"],
      capture_output=True, text=True).stdout.strip())
for p in sys.path:                                   # a 2nd copy here = a shadow install
    for d in glob.glob(os.path.join(p, "numpy-*.dist-info")):
        print("   dist-info:", d)

import numpy
if numpy.__version__ != WANT:
    # Repair in place, then drop numpy from sys.modules so the fresh copy loads. mmcv/mmdet/
    # mmpose have not been imported yet in this kernel, so they will bind to the right numpy.
    print(f"repairing numpy {numpy.__version__} -> {WANT} (no restart) ...")
    print(_sh(f'{sys.executable} -m pip install -q --no-cache-dir --force-reinstall --no-deps "numpy=={WANT}" 2>&1 | tail -3'))
    for _m in [k for k in sys.modules if k == "numpy" or k.startswith("numpy.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    import numpy                                     # the "reloaded" UserWarning here is expected
print("numpy:", numpy.__version__, "@", numpy.__file__)
assert numpy.__version__ == WANT, f"numpy is {numpy.__version__}; re-run Cell 2"

# --- the same py3.12 shims Cell 5 applies ---
if not hasattr(_im.FileFinder, "find_module"):
    def _ff(self, name, path=None):
        s = self.find_spec(name); return s.loader if s is not None else None
    _im.FileFinder.find_module = _ff
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})
if "xtcocotools" not in sys.modules:
    for _n in ("xtcocotools", "xtcocotools.coco", "xtcocotools.cocoeval", "xtcocotools.mask"):
        sys.modules[_n] = types.ModuleType(_n)
    sys.modules["xtcocotools.coco"].COCO = type("COCO", (), {"__init__": lambda s, *a, **k: None})
    sys.modules["xtcocotools.cocoeval"].COCOeval = type("COCOeval", (), {"__init__": lambda s, *a, **k: None})

print("versions:")
for m in ("torch", "mmcv", "mmengine", "mmdet", "mmpose"):
    try:    print(f"   {m:9s}", importlib.import_module(m).__version__)
    except Exception as e: print(f"   {m:9s} <{type(e).__name__}: {str(e)[:80]}>")
print("   transformers:", "REMOVED (good)" if importlib.util.find_spec("transformers") is None
      else "STILL INSTALLED -> run `!pip uninstall -y transformers`, it breaks init_detector on torch 2.3")

def probe(label, fn):
    try:
        fn(); print("  PASS ", label)
    except Exception as e:
        print("  FAIL ", label, "->", type(e).__name__ + ":", str(e)[:160])
        traceback.print_exc(limit=2)

print("critical imports (the ones Cell 5 needs):")
probe("torch cuda", lambda: print("   cuda? ->", importlib.import_module("torch").cuda.is_available()))
probe("from mmdet.apis import init_detector", lambda: __import__("mmdet.apis",  fromlist=["init_detector"]))
probe("from mmpose.apis import init_model",   lambda: __import__("mmpose.apis", fromlist=["init_model"]))
print("\nall PASS -> run Cells 3-8. any FAIL -> paste this whole output.")


In [ ]:
# --- Cell 3: clone RacketVision + download RacketPose checkpoints --------
import os, glob
if not os.path.exists("/content/RacketVision"):
    !git clone --depth 1 https://github.com/OrcustD/RacketVision.git /content/RacketVision
%cd /content/RacketVision/source
!python download_checkpoints.py --module RacketPose

SRC_DIR   = os.getcwd()   # .../RacketVision/source
# Inference configs (confirmed to exist in the repo):
DET_CFG   = os.path.join(SRC_DIR, "RacketPose/configs/detection/rtmdet_m_racket_infer.py")
POSE_CFG  = os.path.join(SRC_DIR, "RacketPose/configs/pose/rtmpose_m_racket_infer.py")
DET_CKPT  = os.path.join(SRC_DIR, "RacketPose/checkpoints/epoch_300.pth")
POSE_CKPT = os.path.join(SRC_DIR, "RacketPose/checkpoints/best_PCK_epoch_90.pth")
for p in (DET_CFG, POSE_CFG, DET_CKPT, POSE_CKPT):
    print("OK  " if os.path.exists(p) else "MISS", p)


In [ ]:
# --- Cell 4: input clip -> 1080p -> frames ------------------------------
# RacketVision was trained at 1080p; test_6 is 4K, so downscale first. We record the
# SOURCE geometry + real fps here (ffprobe, not hardcoded) because Stage 2 has to map
# these 1080p pixel keypoints back onto the SMPL body, which came from a 720p pass.
import json as _json, os, glob, subprocess
from google.colab import files

VIDEO_ID = "test_6"
SRC      = f"/content/{VIDEO_ID}.mp4"

if not os.path.exists(SRC):
    print(f"Upload {VIDEO_ID}.mp4 ...")
    up = files.upload()             # {filename: bytes}
    with open(SRC, "wb") as f:      # write bytes straight to SRC (avoids cwd / '(1)' issues)
        f.write(up[list(up.keys())[0]])

_st = _json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-of", "json",
     "-show_entries", "stream=width,height,r_frame_rate", SRC],
    capture_output=True, text=True).stdout)["streams"][0]
SRC_W, SRC_H = int(_st["width"]), int(_st["height"])
_num, _den = _st["r_frame_rate"].split("/")
FPS = round(float(_num) / float(_den), 3)
print(f"source: {SRC_W}x{SRC_H} @ {FPS} fps")

V1080  = f"/content/{VIDEO_ID}_1080.mp4"
FRAMES = f"/content/frames_{VIDEO_ID}"
os.makedirs(FRAMES, exist_ok=True)
!ffmpeg -y -loglevel error -i {SRC} -vf "scale=-2:1080" -c:v libx264 -crf 18 -an {V1080}
!ffmpeg -y -loglevel error -i {V1080} -q:v 2 {FRAMES}/%05d.jpg
frame_files = sorted(glob.glob(f"{FRAMES}/*.jpg"))
assert frame_files, "no frames extracted -- check the ffmpeg output above"
print("frames:", len(frame_files), "| first:", frame_files[0])


In [ ]:
# --- Cell 5: build models (mmpose top-down demo pattern) ----------------
# --- py3.12 import shims (mmpose.apis drags in old setuptools/pkg_resources) --------
# (a) py3.12 removed importlib FileFinder.find_module, but pkg_resources (imported
#     lazily by mmpose -> get_installed_path) still calls it while declaring Colab's
#     'google' namespace package -> "'FileFinder' object has no attribute 'find_module'".
#     Restore a compatible find_module BEFORE importing mmpose.
import importlib.machinery as _im
if not hasattr(_im.FileFinder, "find_module"):
    def _ff_find_module(self, name, path=None):
        spec = self.find_spec(name)
        return spec.loader if spec is not None else None
    _im.FileFinder.find_module = _ff_find_module
# (b) py3.12 removed pkgutil.ImpImporter, which that same old pkg_resources references.
import pkgutil
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})

# xtcocotools won't build on py3.12; mmpose only needs the NAME importable for top-down
# inference (we never build a COCO dataset). Register a lightweight stub before importing mmpose.
import sys, types
if "xtcocotools" not in sys.modules:
    _pkg = types.ModuleType("xtcocotools")
    _coco = types.ModuleType("xtcocotools.coco")
    _cocoeval = types.ModuleType("xtcocotools.cocoeval")
    _mask = types.ModuleType("xtcocotools.mask")
    class COCO:
        def __init__(self, *a, **k): pass
    class COCOeval:
        def __init__(self, *a, **k): pass
    _coco.COCO = COCO; _cocoeval.COCOeval = COCOeval
    for _fn in ("iou", "decode", "encode", "area", "toBbox", "frPyObjects", "merge"):
        setattr(_mask, _fn, lambda *a, **k: None)
    _pkg.coco, _pkg.cocoeval, _pkg.mask = _coco, _cocoeval, _mask
    sys.modules.update({"xtcocotools": _pkg, "xtcocotools.coco": _coco,
                        "xtcocotools.cocoeval": _cocoeval, "xtcocotools.mask": _mask})
    print("xtcocotools stub registered")

# numpy guard: mmcv/mmdet ops are compiled against numpy 1.x. If numpy isn't 1.26.4,
# the imports below crash with "numpy.dtype size changed". Fail fast with a clear fix.
import numpy as _np
if _np.__version__ != "1.26.4":
    raise SystemExit("numpy is " + _np.__version__ + " but the stack needs 1.26.4. Run Cell 2 "
                     "(it pins numpy + auto-restarts), then run Cell 2c to verify, then Cells 3-8.")

import functools, torch
torch.load = functools.partial(torch.load, weights_only=False)   # mmengine ckpts; harmless on torch 2.3
from mmdet.apis import init_detector, inference_detector
from mmpose.apis import init_model as init_pose, inference_topdown
from mmpose.structures import merge_data_samples
from mmpose.utils import adapt_mmdet_pipeline

print("torch:", torch.__version__, "| cuda?", torch.cuda.is_available())
DEV = "cuda:0" if torch.cuda.is_available() else "cpu"

detector = init_detector(DET_CFG, DET_CKPT, device=DEV)
detector.cfg = adapt_mmdet_pipeline(detector.cfg)
poser    = init_pose(POSE_CFG, POSE_CKPT, device=DEV)

meta = poser.dataset_meta
n = meta.get("num_keypoints", 0)
print("num_keypoints:", n)
print("keypoint order:", [meta["keypoint_id2name"][i] for i in range(n)])
# expected: ['top', 'bottom', 'handle', 'left', 'right']


In [ ]:
# --- Cell 6: inference loop -> clean 2D json ---------------------------
# v4 -- RECALL FIRST. v3 ran DET_THR=0.30 and hit only 16/189 frames on test_6, yet the
# keypoints were spot-on WHENEVER it fired (a 0.31-score overhead box fit the racket
# perfectly). So the RTMDet detector is the bottleneck, not the RTMPose head. Two changes:
#   1. DET_THR 0.30 -> 0.05: be permissive here, filter on det_score/keypoint_scores later.
#   2. keep the TOP-K boxes per frame WITH keypoints for each, not just argmax. Choosing
#      the right one is a temporal-continuity decision best made locally -- this way a bad
#      pick costs a rerun of a local script, not another Colab install gauntlet.
import mmcv, numpy as np, json

KP_NAMES = ["top", "bottom", "handle", "left", "right"]   # repo order (confirmed)
CAT_ID   = 0        # badminton in RacketVision's detector (0=badminton, 1=tabletennis, 2=tennis)
DET_THR  = 0.05     # deliberately low -- see above
MAX_RATIO = 0.5     # drop boxes wider/taller than half the frame (their guard)
TOPK     = 3        # candidate boxes kept per frame (RTMPose is cheap; detector recall is the risk)
STRIDE   = 1        # set >1 for a quick subsampled pass (e.g. 5) if you're on CPU

FRAME_H, FRAME_W = mmcv.imread(frame_files[0]).shape[:2]   # the 1080p frames we infer on

results = []
idxs = list(range(0, len(frame_files), STRIDE))
for k, i in enumerate(idxs):
    img = mmcv.imread(frame_files[i])
    H, W = img.shape[:2]
    det = inference_detector(detector, img)
    inst = det.pred_instances.cpu().numpy()
    keep = (inst.labels == CAT_ID) & (inst.scores >= DET_THR)
    bb, sc = inst.bboxes[keep], inst.scores[keep]
    if len(bb):
        w = (bb[:, 2] - bb[:, 0]) / W
        h = (bb[:, 3] - bb[:, 1]) / H
        ok = (w < MAX_RATIO) & (h < MAX_RATIO)
        bb, sc = bb[ok], sc[ok]
    order = list(np.argsort(-sc)[:TOPK]) if len(sc) else []
    cands = []
    for b in order:
        bbox = bb[b:b + 1]                            # (1,4) xyxy
        pr = inference_topdown(poser, img, bbox, bbox_format="xyxy")
        ds = merge_data_samples(pr)
        cands.append({"bbox": bbox[0].tolist(), "det_score": float(sc[b]),
                      "keypoints": ds.pred_instances.keypoints[0].tolist(),
                      "keypoint_scores": ds.pred_instances.keypoint_scores[0].tolist()})
    # flat fields = the argmax candidate (unchanged contract); `cands` = all of them.
    best = cands[0] if cands else {"bbox": None, "det_score": None,
                                   "keypoints": None, "keypoint_scores": None}
    results.append({"frame": i, **best, "cands": cands})
    if k % 50 == 0:
        print(k, "/", len(idxs))

# Geometry is part of the contract: `keypoints` are pixels in a FRAME_W x FRAME_H frame,
# which is the source downscaled to 1080p. Stage 2 needs both to line the racket up with
# the SMPL body (extracted from a different downscale of the same clip).
out = {"video_id": VIDEO_ID, "fps": FPS, "stride": STRIDE,
       "frame_size": [FRAME_W, FRAME_H], "source_size": [SRC_W, SRC_H],
       "source": "RacketVision RTMDet+RTMPose (pretrained)",
       "det": {"cat_id": CAT_ID, "score_thr": DET_THR,
               "max_box_ratio": MAX_RATIO, "topk": TOPK},
       "keypoint_names": KP_NAMES, "num_frames": len(results), "frames": results}
JSON_OUT = f"/content/{VIDEO_ID}.racket2d.json"
json.dump(out, open(JSON_OUT, "w"))
hit = sum(1 for r in results if r["keypoints"])
print(f"racket detected in {hit}/{len(results)} sampled frames -> {JSON_OUT}")
print("hit rate if we had thresholded at:")
for t in (0.05, 0.10, 0.20, 0.30, 0.50):
    n = sum(1 for r in results if r["det_score"] is not None and r["det_score"] >= t)
    print(f"   det >= {t:.2f}: {n}/{len(results)}  ({100 * n / len(results):.0f}%)")


In [ ]:
# --- Cell 7: overlay video (visual sanity check) -----------------------
# Draws the chosen (top-scoring) candidate solid, and the other TOPK-1 candidates as thin
# gray boxes -- so a wrong pick is visible instead of silently becoming the answer.
import cv2, numpy as np, os
COLORS = {"top": (0,255,0), "bottom": (0,200,255), "handle": (255,80,80),
          "left": (255,0,255), "right": (0,255,255)}   # BGR
name2idx = {n: i for i, n in enumerate(KP_NAMES)}
OVR = f"/content/overlay_{VIDEO_ID}"
os.makedirs(OVR, exist_ok=True)

for j, i in enumerate(idxs):
    img = cv2.imread(frame_files[i])
    r = results[j]
    for c in r.get("cands", [])[1:]:                     # runner-up boxes, context only
        x1, y1, x2, y2 = (int(round(v)) for v in c["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (120,120,120), 1)
        cv2.putText(img, f"{c['det_score']:.2f}", (x1, max(14, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (120,120,120), 1)
    if r["keypoints"]:
        kp = np.array(r["keypoints"])
        # cv2 rejects numpy ints in point tuples -- cast to python int.
        def P(name):
            x, y = kp[name2idx[name]]
            return (int(round(float(x))), int(round(float(y))))
        cv2.line(img, P("handle"), P("top"), (255,255,255), 3)   # shaft / long axis
        cv2.line(img, P("left"), P("right"), (200,200,200), 2)   # head width
        for name in KP_NAMES:
            cv2.circle(img, P(name), 6, COLORS[name], -1)
        kmin = min(r["keypoint_scores"])
        cv2.putText(img, f"det {r['det_score']:.2f}  kp>={kmin:.2f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
    else:
        cv2.putText(img, "no racket", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
    cv2.imwrite(f"{OVR}/{j:05d}.jpg", img)

OVR_MP4 = f"/content/{VIDEO_ID}_racket_overlay.mp4"
os.system(f'ffmpeg -y -loglevel error -framerate {max(1, round(FPS / STRIDE))} '
          f'-i {OVR}/%05d.jpg -c:v libx264 -pix_fmt yuv420p {OVR_MP4}')
print("overlay ->", OVR_MP4)


In [ ]:
# --- Cell 8: download outputs ------------------------------------------
from google.colab import files
files.download(JSON_OUT)     # test_6.racket2d.json  -> save to data/racket/
files.download(OVR_MP4)      # overlay to eyeball the detection


## What to send back
1. **`test_6.racket2d.json`** -> save to `data/racket/test_6.racket2d.json`.
2. **`test_6_racket_overlay.mp4`** -> so we can see the 5 keypoints tracking through the swing.
3. The **Cell 5 keypoint-order** line and the whole **Cell 6 hit-rate** block (the
   `det >= x` table tells us where to cut).

**v4 note:** the detector, not the pose head, is the weak link -- v3 at `DET_THR=0.30`
found the racket in only 16/189 frames of test_6, while the keypoints it *did* produce
sat exactly on the racket (even a 0.31-score box). v4 runs at `DET_THR=0.05` and keeps
the **top 3 boxes per frame with keypoints for each** (`frames[i].cands`), so the
pick-the-right-box decision moves local, where it costs nothing to redo.

## If a cell fights back
- **`NameError: name 'nn' is not defined`** (raised from `transformers/integrations/accelerate.py`
  while `init_detector` builds the model): mmdet imports `transformers` for its language models,
  and Colab's transformers requires torch >= 2.4 -- on our pinned torch 2.3.1 it half-disables
  itself and then dies with a NameError, which mmdet's `except ImportError` cannot catch.
  Fix: `!pip uninstall -y transformers` (Cell 2 step 4b does this), then Runtime > Restart session
  and run Cell 2c, 3, 4, 5. If some other package turns out to need it, install the last
  torch-2.3-compatible release instead: `!pip install {C} "transformers==4.44.2"`.
- **`numpy.dtype size changed ... Expected 96 ... got 88`** OR **`cannot import name '_center'
  from numpy._core.umath`:** numpy is 2.x at runtime but mmcv/mmdet are compiled against numpy 1.x.
  **Cell 2c repairs this in-kernel** (pip force-reinstall 1.26.4 + drop numpy from `sys.modules`)
  -- just run it. Note a kernel that ran Cell 2 has ALREADY imported numpy 2.x, so its in-kernel
  version stays stale until the restart or the repair; trust Cell 2c's "numpy ON DISK" line.
- **mmcv wheel 404 / not found:** the torch2.3.0/cu121/cp312 mmcv wheel is missing -> tell me and
  I'll switch the pin (torch 2.4.1 or a source build).
- **`No module named 'xtcocotools'`:** it won't build on py3.12 -- Cell 5 registers a stub instead
  (top-down inference never builds a COCO dataset). If a NEW `chumpy` import appears, paste it and I'll stub that too.
- **`'FileFinder' object has no attribute 'find_module'`:** py3.12 deleted that method, but setuptools'
  `pkg_resources` (pulled in when mmpose builds its inferencer registry) still calls it while declaring
  Colab's `google` namespace package. Cell 5 restores a compatible `find_module` before importing mmpose.
- **`scope mmpose exists in runner registry`:** mmpose was imported twice in one kernel (e.g. after purging
  it from sys.modules). Runtime > Restart session, then run Cells 2c, 3-8 once, in order.
- **general "which import broke?":** run **Cell 2c (DEBUG PROBE)** -- it imports each module one at a time
  and prints PASS/FAIL + a 2-line trace, so we see the exact culprit without a full re-run.
- **VM recycled / everything gone:** Colab wipes `/content` + all installs after a few hours idle.
  Re-run 1 -> 2 -> 2c -> 3-8. Caching the built env to Drive is a TODO.
- **Fallback:** we already have `tools/detect_racket.py` (COCO box detector) that runs locally if the
  RacketVision stack stays uncooperative.

## Next (local, after this)
- **Stage 2:** lift these 2D keypoints to a 3D racket segment anchored at the SMPL hand
  (grip idx24 / head idx25), append to `data/skeleton/test_6.skeleton.json`. The JSON carries
  `frame_size` + `source_size` so the 1080p pixels can be mapped onto the 720p SMPL pass.
- **Stage 3:** show the racket on the Blender twin + compare viewer.
